# EfficientNet on HAM10000 — Kaggle GPU training

Kaggle twin of `skin/colab_training.ipynb`. Same two-phase transfer-learning recipe
and the same outputs (`*_best.pt`, `model_config.json`, `classes.json`, plots), but it
reads the **local Kaggle HAM10000 dataset** and trains on a **GPU** instead of streaming
from Hugging Face and training on a Colab TPU.

**How to run on Kaggle**

1. Create a new Notebook (Kaggle → Code → New Notebook) and upload this file
   (File → Import Notebook), or paste the cells in.
2. **Add the data**: right sidebar → *Add Input* → search **"Skin Cancer MNIST: HAM10000"**
   (by *kmader*) → add it. It mounts read-only at
   `/kaggle/input/skin-cancer-mnist-ham10000/`.
3. **Turn on the GPU**: right sidebar → *Settings* → *Accelerator* → **GPU T4 x2**
   (use **T4**, not P100 — Kaggle's PyTorch build dropped support for the older P100's
   sm_60 compute capability, so it errors out; the T4 is sm_75 and works. The notebook
   uses a single GPU, so the second T4 just sits idle).
4. *Run All*. Training writes everything to `/kaggle/working/results/`.
5. When done, click **Save Version** (top-right) so `/kaggle/working/` is stored with the
   run; you can then download the contents of `results/` from the version's *Output* tab.

The default backbone is **EfficientNetV2-S @ 384px** (change `MODEL_NAME` in the config
cell for `efficientnet_b3`, `convnext_small`, `swin_v2_t`, `vit_b_16`, …; the head-swap is
generic). Progress is printed and also appended to `results/progress.log`, so a run is
monitorable even if the tab disconnects.

### Knobs to experiment with (all in the config cell)

Transfer learning happens in two phases — train the new head briefly (backbone frozen),
then fine-tune the whole network at a low learning rate — plus an optional third phase that
rebalances the classifier (see *Combating class imbalance* in the README).

- `MODEL_NAME` — swap the backbone. Any torchvision classifier works.
- `IMG_SIZE` — input resolution (V2-S is native at 384; `efficientnet_b3` at 300).
- `FT_LR` / `HEAD_LR` — fine-tuning learning rates.
- `FT_EPOCHS` / `EARLY_STOP_PATIENCE` — max epochs and patience before early-stopping.
- `LOGIT_ADJUST_TAU` — strength of the logit-adjusted loss (the imbalance corrector for
  phases 1–2). `1.0` is standard; `0` falls back to inverse-frequency class-weighted CE.
- `CRT_EPOCHS` / `CRT_LR` — epochs/LR of phase-3 decoupled classifier re-training on a
  class-balanced sampler. `0` skips it.
- `TRAIN_CAP` — per-class cap on training images (default uncapped; lower it to rebalance
  the heavily melanocytic-nevi-dominated training set the cruder, data-discarding way).
- `BATCH_SIZE` — lower to 16 if you hit CUDA OOM at 300px, raise to 48 on a 16GB card.

The split is **grouped by `lesion_id`** so multiple images of the same lesion never land in
both train and val (HAM10000 has ~2 images per lesion on average — an ungrouped split
leaks and inflates the score).

## 1. Environment

In [ ]:
# Kaggle ships torch, torchvision, scikit-learn, matplotlib, pillow — nothing to install.
import os
import json
import time
import csv
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

IN_KAGGLE = Path("/kaggle/input").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
USE_AMP = device.type == "cuda"  # mixed precision: big speed/memory win on GPU, no-op elsewhere

# Mixed-precision helpers, tolerant of the torch.amp (>=2.3) vs torch.cuda.amp API split.
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler():
        return _GradScaler(device.type, enabled=USE_AMP)
    def amp_ctx():
        return _autocast(device.type, enabled=USE_AMP)
except Exception:  # older torch
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler():
        return _GradScaler(enabled=USE_AMP)
    def amp_ctx():
        return _autocast(enabled=USE_AMP)

print(f"PyTorch {torch.__version__}")
print(f"Kaggle: {IN_KAGGLE}  device: {device}  AMP: {USE_AMP}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Fail fast on an unsupported GPU. Kaggle's P100 is sm_60, but recent PyTorch
    # CUDA builds only ship kernels for sm_70+ — without this check the run would
    # crash with a cryptic error deep inside the first forward pass instead.
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    if supported and arch not in supported:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(0)} ({arch}) is not supported by this "
            f"PyTorch build (supports {supported}). On Kaggle, switch "
            "Settings -> Accelerator -> GPU T4 x2 (the T4 is sm_75) and Run All again."
        )
elif IN_KAGGLE:
    print("WARNING: no GPU detected. Settings -> Accelerator -> GPU T4 x2, then Run All again.")

## 2. Configuration

In [ ]:
# --- paths ------------------------------------------------------------------
# The Kaggle HAM10000 dataset mounts read-only here. INPUT_DIR is searched
# recursively, so the exact sub-folder layout (HAM10000_images_part_1/2, etc.)
# does not matter.
if IN_KAGGLE:
    INPUT_DIR = Path("/kaggle/input/skin-cancer-mnist-ham10000")
    OUT_DIR = Path("/kaggle/working/results")  # saved with the notebook version
else:
    # Running this notebook locally: point INPUT_DIR at an unzipped copy of the
    # Kaggle dataset (the folder containing HAM10000_metadata.csv + the images).
    INPUT_DIR = Path.cwd() / "data" / "ham10000_raw"
    OUT_DIR = Path.cwd() / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- label map --------------------------------------------------------------
# Kaggle's metadata uses abbreviated dx codes; map them to the full class names
# the rest of the project (classes.json, the app, the analysis script) expects.
# sorted(DX_TO_CLASS.values()) reproduces the exact class order in results/classes.json,
# so a model trained here is drop-in compatible with skin/03_app.py.
DX_TO_CLASS = {
    "akiec": "actinic_keratoses",
    "bcc": "basal_cell_carcinoma",
    "bkl": "benign_keratosis-like_lesions",
    "df": "dermatofibroma",
    "nv": "melanocytic_nevi",
    "mel": "melanoma",
    "vasc": "vascular_lesions",
}

# --- hyperparameters (same knobs as the Colab notebook) ---------------------
MODEL_NAME = "efficientnet_b3"     # tuned on b3 in sweep ukf4v67c; proven best architecture
IMG_SIZE = 300                     # EfficientNet-B3 native resolution
VAL_RESIZE = round(IMG_SIZE * 256 / 224)  # same resize/crop ratio as the 224 recipe
BATCH_SIZE = 32                    # lower to 16 if CUDA OOM at 300px; raise to 48 on 16GB
NUM_WORKERS = min(4, os.cpu_count() or 2)  # Kaggle GPU VMs have ~4 CPU cores

HEAD_EPOCHS = 3
FT_EPOCHS = 40
EARLY_STOP_PATIENCE = 6            # stop phase 2 after this many epochs with no val gain
HEAD_LR = 5e-4                     # sweep ukf4v67c: lower head LR scored better (r=-0.51)
FT_LR = 1.5e-4                     # sweep ukf4v67c: dominant knob, best ~1-3e-4 (r=+0.86)

# --- class-imbalance handling (see README "Combating class imbalance") ------
# HAM10000 is ~67% melanocytic nevi, so a plain cross-entropy run is biased
# toward the majority class. Two modern, complementary correctors are wired in:
#
#   1. Logit adjustment (Menon et al., ICLR 2021). Adds tau * log(class prior)
#      to the logits *inside the loss*. At inference the raw logits are then
#      Bayes-corrected for the label distribution, with no app/analysis change.
#      It replaces inverse-frequency class weighting (combining the two
#      double-corrects), so when tau > 0 the weighted CE is switched off.
#      tau = 1.0 is the standard, theoretically-motivated setting; 0 disables it.
#
#   2. Decoupled classifier re-training / cRT (Kang et al., ICLR 2020). After the
#      natural-distribution fine-tune, freeze the backbone and re-train ONLY the
#      head for a few epochs with a class-balanced sampler, rebalancing the
#      decision boundary without disturbing the learned features. CRT_EPOCHS = 0
#      disables this phase. The cRT phase uses plain (unweighted) CE because the
#      balanced sampler already equalises the classes.
#
# The two are independent; set one of them to 0 to ablate. Phase 3 only replaces
# the checkpoint if it beats phase 2's best val balanced accuracy, so enabling
# cRT can never make the saved model worse.
LOGIT_ADJUST_TAU = 1.0             # sweep ukf4v67c: tau 1.0-1.5 beat tau=0 (r=+0.43)
CRT_EPOCHS = 3                     # sweep ukf4v67c: low impact; small value kept
CRT_LR = 1e-3                      # LR for the cRT head (frozen backbone, so a head-sized LR)

# Split / sampling
SEED = 42
VAL_FRACTION = 0.15                # share of each class's lesions held out for validation
VAL_CAP = 200                      # cap val images per class for a clean, comparable metric
TRAIN_CAP = 100_000               # per-class train cap (100k = effectively uncapped)

# Skip the run if a checkpoint for this MODEL_NAME already exists. Flip to retrain.
FORCE_RETRAIN = True               # retrain the swept-best config from scratch

# --- Weights & Biases experiment tracking -----------------------------------
# When True, loss / val balanced-accuracy stream live to wandb.ai during training
# (plus GPU/CPU/memory system metrics), so you can watch the run from anywhere —
# even if this Kaggle tab disconnects. See the "Experiment tracking" cell below
# for how the API key is read from Kaggle Secrets (never hard-coded).
USE_WANDB = True
WANDB_PROJECT = "skin-lesion"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Model: {MODEL_NAME} @ {IMG_SIZE}px  batch {BATCH_SIZE}  workers {NUM_WORKERS}")
print(f"Head: {HEAD_EPOCHS} epochs @ LR={HEAD_LR}")
print(f"Fine-tune: up to {FT_EPOCHS} epochs @ LR={FT_LR}, early stop patience {EARLY_STOP_PATIENCE}")
print(f"Imbalance: logit-adjust tau={LOGIT_ADJUST_TAU}  cRT epochs={CRT_EPOCHS} @ LR={CRT_LR}")
print(f"Output dir: {OUT_DIR}")

## 2b. Experiment tracking (Weights & Biases)


In [ ]:
# --- Weights & Biases setup (safe no-op when USE_WANDB = False) --------------
# Auth on Kaggle WITHOUT putting your key in the notebook:
#   Add-ons -> Secrets -> add WANDB_API_KEY (from https://wandb.ai/authorize),
#   then tick this notebook to grant access. Running locally, an env var
#   WANDB_API_KEY (or a prior `wandb login`) is picked up automatically.
wandb = None
if USE_WANDB:
    try:
        import wandb
    except ImportError:
        import subprocess, sys as _sys
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
        import wandb

    _key = os.environ.get("WANDB_API_KEY")
    if _key is None and IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            _key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception as e:
            print(f"No WANDB_API_KEY secret available ({e}).")
    if _key:
        _key = _key.strip()  # guard against a stray newline/space in the secret
    if _key and len(_key) >= 40:
        wandb.login(key=_key)
        print(f"wandb: logged in ({len(_key)}-char key), live tracking ON.")
    elif _key:
        raise ValueError(
            f"WANDB_API_KEY looks invalid: {len(_key)} characters (a real key is "
            "40+). Re-copy the FULL key from https://wandb.ai/authorize into the "
            "Kaggle secret WANDB_API_KEY — a partial paste is the usual cause.")
    else:
        print("wandb: no API key found -> tracking OFF (training still runs).")
        USE_WANDB = False


## 3. Build the train / val split

Reads `HAM10000_metadata.csv`, indexes every image file by `image_id`, then makes a
**seeded, lesion-grouped, per-class** split: all images of one lesion go to the same side,
~`VAL_FRACTION` of each class's lesions are held out for validation (capped at `VAL_CAP`
images/class), and the remaining training images are capped per class at `TRAIN_CAP`.

In [ ]:
# 0) locate the dataset. The mount slug / sub-folder layout can vary, so find the
#    HAM10000_metadata.csv anywhere under /kaggle/input and use its folder as root.
search_roots = [INPUT_DIR]
if IN_KAGGLE:
    search_roots.append(Path("/kaggle/input"))
meta_csv = None
for root in search_roots:
    if root.exists():
        meta_csv = next(root.rglob("HAM10000_metadata*.csv"), None)
        if meta_csv is not None:
            break
if meta_csv is None:
    listing = "\n".join(f"  {p}" for p in sorted(Path('/kaggle/input').glob('*'))) \
        if Path('/kaggle/input').exists() else "  (/kaggle/input does not exist)"
    raise FileNotFoundError(
        "Could not find HAM10000_metadata.csv. Add the dataset via the right "
        "sidebar -> Add Input -> 'Skin Cancer MNIST: HAM10000' (by kmader).\n"
        f"Currently mounted under /kaggle/input:\n{listing}"
    )
INPUT_DIR = meta_csv.parent
print(f"Dataset root: {INPUT_DIR}")

# 1) index every image file by its image_id (extensions/case vary across mirrors)
img_paths = {}
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    for p in INPUT_DIR.rglob(ext):
        img_paths.setdefault(p.stem, p)
print(f"Indexed {len(img_paths)} image files under {INPUT_DIR}")
assert img_paths, f"Found metadata but no image files under {INPUT_DIR} — check the dataset is complete."

# 2) read the metadata (image_id, lesion_id, dx, ...)
with open(meta_csv, newline="") as f:
    rows = list(csv.DictReader(f))
print(f"Metadata: {len(rows)} rows ({meta_csv.name})")

# 3) group images by lesion so the same lesion can't appear in both splits
lesion_imgs = defaultdict(list)   # lesion_id -> [image_id, ...]
lesion_dx = {}                    # lesion_id -> dx code
for r in rows:
    iid, lid, dx = r["image_id"], r["lesion_id"], r["dx"].lower()
    if iid not in img_paths:
        continue  # metadata row with no matching image file
    lesion_imgs[lid].append(iid)
    lesion_dx[lid] = dx

classes = sorted(DX_TO_CLASS[dx] for dx in set(lesion_dx.values()))
class_to_idx = {c: i for i, c in enumerate(classes)}
assert len(classes) == 7, f"Expected 7 classes, got {classes}"

lesions_by_class = defaultdict(list)
for lid, dx in lesion_dx.items():
    lesions_by_class[DX_TO_CLASS[dx]].append(lid)

# 4) per-class grouped split
rng = random.Random(SEED)
train_samples, val_samples = [], []   # each: (Path, class_idx)
for cls in classes:
    lesions = lesions_by_class[cls]
    rng.shuffle(lesions)
    total_imgs = sum(len(lesion_imgs[l]) for l in lesions)
    val_target = min(VAL_CAP, max(1, round(VAL_FRACTION * total_imgs)))
    val_n, train_n = 0, 0
    for lid in lesions:
        imgs = [(img_paths[iid], class_to_idx[cls]) for iid in lesion_imgs[lid]]
        if val_n < val_target:
            val_samples.extend(imgs)
            val_n += len(imgs)
        elif train_n < TRAIN_CAP:
            take = imgs[: max(0, TRAIN_CAP - train_n)]
            train_samples.extend(take)
            train_n += len(take)

rng.shuffle(train_samples)
print(f"Classes: {classes}")
print(f"Train: {len(train_samples)} images   Val: {len(val_samples)} images")
for cls in classes:
    ci = class_to_idx[cls]
    tr = sum(1 for _, y in train_samples if y == ci)
    va = sum(1 for _, y in val_samples if y == ci)
    print(f"  {cls:30s} train {tr:5d}  val {va:4d}")

## 4. Data loaders

In [ ]:
class SkinDataset(Dataset):
    """ImageFolder-compatible dataset over an explicit (path, label) list.

    Exposes `.samples` and `.classes` so the class-weight helper and the rest of
    the notebook work unchanged. Reads images straight from the read-only Kaggle
    input — the HAM10000 jpgs are already 600x450, large enough for 384px inputs,
    so there is no need to copy or pre-resize ~2.6GB of files."""
    def __init__(self, samples, transform, classes):
        self.samples = samples
        self.transform = transform
        self.classes = classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),  # lesions have no canonical orientation
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(VAL_RESIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = SkinDataset(train_samples, train_tf, classes)
val_ds = SkinDataset(val_samples, val_tf, classes)

_pin = device.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=_pin, persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=_pin, persistent_workers=NUM_WORKERS > 0,
)
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

## 5. Model setup

In [ ]:
def class_weights(samples, n_classes):
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)


def log_class_prior(samples, n_classes):
    """log P(y) over the training set — the offset logit adjustment subtracts."""
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    prior = counts / counts.sum()
    return torch.log(torch.tensor(prior, dtype=torch.float32).clamp_min(1e-12))


class LogitAdjustedLoss(nn.Module):
    """Logit-adjusted cross-entropy (Menon et al., ICLR 2021).

    Trains on ``logits + tau * log_prior``, which is equivalent to enforcing a
    per-class margin proportional to the label frequency. Because the prior is
    added during training, the *raw* logits at inference are already corrected
    for the class imbalance — so the app and analysis need no change. This is an
    alternative to inverse-frequency class weighting, not an addition to it."""
    def __init__(self, log_prior, tau):
        super().__init__()
        self.register_buffer("adj", tau * log_prior)

    def forward(self, logits, target):
        return F.cross_entropy(logits + self.adj, target)


def balanced_sampler(samples, n_classes):
    """A WeightedRandomSampler that draws each class with equal probability, for
    the cRT phase. Sampling (rather than capping) keeps every image available
    while equalising how often each class is seen per epoch."""
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    per_class_w = 1.0 / np.maximum(counts, 1)
    sample_w = [per_class_w[y] for _, y in samples]
    return WeightedRandomSampler(sample_w, num_samples=len(samples), replacement=True)


def replace_head(model, n_classes):
    """Swap the final classification layer for a fresh n_classes one.

    Different torchvision families name the head differently, so we locate the
    last nn.Linear generically. Works for EfficientNet/ConvNeXt (.classifier),
    Swin (.head), ViT (.heads), ResNet (.fc). Mirrors skin/skin_model.replace_head."""
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is None:
            continue
        if isinstance(head, nn.Linear):
            setattr(model, head_attr, nn.Linear(head.in_features, n_classes))
            return model
        last_linear_name = None
        for name, m in head.named_modules():
            if isinstance(m, nn.Linear):
                last_linear_name = name
        if last_linear_name is not None:
            parent = head
            *path, leaf = last_linear_name.split(".")
            for p in path:
                parent = getattr(parent, p)
            in_features = getattr(parent, leaf).in_features
            setattr(parent, leaf, nn.Linear(in_features, n_classes))
            return model
    raise ValueError(f"Could not find a classifier head on {type(model).__name__}")


def freeze_backbone(model, freeze=True):
    """Phase-1 helper: freeze everything except the classification head."""
    head_params = set()
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is not None:
            head_params.update(id(p) for p in head.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_params) if freeze else True


def head_parameters(model):
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is not None:
            return head.parameters()
    return model.parameters()


model = models.get_model(MODEL_NAME, weights="IMAGENET1K_V1")
model = replace_head(model, len(classes))
model.to(device)

# Phase 1/2 loss: logit adjustment if enabled, else inverse-frequency weighted CE.
# (Using both would double-correct the imbalance, so they are mutually exclusive.)
if LOGIT_ADJUST_TAU > 0:
    criterion = LogitAdjustedLoss(log_class_prior(train_samples, len(classes)), LOGIT_ADJUST_TAU).to(device)
    print(f"Loss: logit-adjusted CE (tau={LOGIT_ADJUST_TAU})")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_samples, len(classes)).to(device))
    print("Loss: inverse-frequency class-weighted CE")
history = []

CKPT_PATH = OUT_DIR / f"{MODEL_NAME}_best.pt"

# the Gradio app (skin/03_app.py) reads this to rebuild the right architecture
with open(OUT_DIR / "model_config.json", "w") as f:
    json.dump({"model": MODEL_NAME, "img_size": IMG_SIZE, "checkpoint": CKPT_PATH.name}, f, indent=2)

print(f"Model: {MODEL_NAME}, output classes: {len(classes)}")
print(f"Checkpoint will be saved to: {CKPT_PATH}")

## 6. Evaluation function

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad(), amp_ctx():
        for x, y in loader:
            logits = model(x.to(device, non_blocking=True))
            preds.append(logits.argmax(1).cpu().numpy())
            targets.append(y.numpy())
    preds, targets = np.concatenate(preds), np.concatenate(targets)
    return preds, targets, balanced_accuracy_score(targets, preds)

## 7. Training

Up to three phases: (1) head-only warm-up, (2) a full fine-tune at a low LR with early
stopping, and — when `CRT_EPOCHS > 0` — (3) **decoupled classifier re-training (cRT)**:
the backbone is frozen again and only the head is re-trained on a class-balanced sampler,
rebalancing the decision boundary for the imbalanced classes. The imbalance is also handled
inside phases 1–2 by the **logit-adjusted loss** (`LOGIT_ADJUST_TAU`). See the README's
*Combating class imbalance* section. Mixed precision (AMP) is on automatically on GPU;
progress prints every `LOG_EVERY` batches and is mirrored to `results/progress.log`.

In [ ]:
LOG_EVERY = 20  # batches between progress prints


def log_line(msg):
    """Print progress and append it to results/progress.log so a run stays
    monitorable even if the Kaggle tab disconnects."""
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {msg}", flush=True)
    with open(OUT_DIR / "progress.log", "a") as f:
        f.write(f"[{stamp}] {msg}\n")


def train_epochs(model, loader, val_loader, criterion, optimizer, epochs, tag, history,
                 scaler, patience=None, best=-1.0):
    """Train for up to `epochs`. If `patience` is set, stop early once validation
    balanced accuracy hasn't improved for that many consecutive epochs. The best
    checkpoint is saved whenever it improves, so an early stop (or a disconnect)
    always leaves the best weights on disk. `best` lets a later phase continue from
    an earlier phase's best score rather than resetting it."""
    since_improve = 0
    n_batches = len(loader)
    for epoch in range(epochs):
        model.train()
        running, n = 0.0, 0
        t0 = time.time()
        for i, (x, y) in enumerate(loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            with amp_ctx():
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * len(y)
            n += len(y)
            if (i + 1) % LOG_EVERY == 0 or (i + 1) == n_batches:
                per_batch = (time.time() - t0) / (i + 1)
                eta = per_batch * (n_batches - i - 1)
                print(f"  [{tag} {epoch + 1}/{epochs}] batch {i + 1}/{n_batches}  "
                      f"loss {running / n:.4f}  ~{eta:.0f}s left in epoch", flush=True)
        _, _, bal_acc = evaluate(model, val_loader)
        epoch_loss = running / n
        history.append({"phase": tag, "epoch": epoch, "loss": epoch_loss, "val_bal_acc": bal_acc})
        if USE_WANDB and wandb is not None:
            # len(history) is a single counter that increases across all three
            # phases, so the live curve reads left-to-right head -> ft -> crt.
            wandb.log({"train_loss": epoch_loss, "val_bal_acc": bal_acc,
                       "phase": tag, "phase_epoch": epoch}, step=len(history))
        log_line(f"[{tag}] epoch {epoch + 1}/{epochs}  loss {epoch_loss:.4f}  "
                 f"val bal-acc {bal_acc:.3f}  ({time.time() - t0:.0f}s)")
        if bal_acc > best:
            best = bal_acc
            since_improve = 0
            torch.save(model.state_dict(), CKPT_PATH)
            if USE_WANDB and wandb is not None and wandb.run is not None:
                wandb.run.summary["best_val_bal_acc"] = best
            log_line(f"  -> new best checkpoint saved (val bal-acc {bal_acc:.3f})")
        else:
            since_improve += 1
            if patience is not None and since_improve >= patience:
                log_line(f"  -> early stop: no val improvement in {patience} epochs (best {best:.3f})")
                break
    return best


# Compute saver: skip the run if this model is already trained in OUT_DIR.
LOG_PATH = OUT_DIR / "training_log.json"
already_trained = CKPT_PATH.exists() and LOG_PATH.exists() and not FORCE_RETRAIN
if already_trained:
    prev = json.load(open(LOG_PATH))
    if prev.get("model") == MODEL_NAME:
        history = prev["history"]
        best = prev["best_val_bal_acc"]
        log_line(f"Found existing {MODEL_NAME} (val bal-acc {best:.3f}) — skipping training. "
                 f"Set FORCE_RETRAIN=True to retrain.")
    else:
        already_trained = False  # log belongs to a different model; train fresh

if not already_trained:
    scaler = make_scaler()
    log_line(f"Training started: {MODEL_NAME} @ {IMG_SIZE}px, "
             f"{len(train_ds)} train / {len(val_ds)} val images on {device}")

    if USE_WANDB and wandb is not None:
        wandb.init(project=WANDB_PROJECT, name=MODEL_NAME, config={
            "model": MODEL_NAME, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
            "head_epochs": HEAD_EPOCHS, "ft_epochs": FT_EPOCHS,
            "head_lr": HEAD_LR, "ft_lr": FT_LR,
            "crt_epochs": CRT_EPOCHS, "crt_lr": CRT_LR,
            "logit_adjust_tau": LOGIT_ADJUST_TAU,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "val_cap": VAL_CAP, "train_cap": TRAIN_CAP, "seed": SEED,
        })

    # Phase 1: freeze backbone, train head only
    log_line("=== Phase 1: head training ===")
    freeze_backbone(model, freeze=True)
    opt = torch.optim.AdamW(head_parameters(model), lr=HEAD_LR)
    best = train_epochs(model, train_loader, val_loader, criterion, opt, HEAD_EPOCHS,
                        "head", history, scaler)

    # Phase 2: unfreeze everything, fine-tune at low LR with early stopping
    log_line("=== Phase 2: full fine-tuning ===")
    freeze_backbone(model, freeze=False)
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR)
    best = train_epochs(model, train_loader, val_loader, criterion, opt, FT_EPOCHS,
                        "ft", history, scaler, patience=EARLY_STOP_PATIENCE, best=best)

    # Phase 3: decoupled classifier re-training (cRT). Restore phase-2's best
    # weights, freeze the backbone, and re-train only the head on a class-balanced
    # sampler with plain CE. This rebalances the decision boundary on top of the
    # features learned from the natural distribution. It carries `best` forward, so
    # the checkpoint is only overwritten if cRT actually improves val balanced
    # accuracy — enabling it can never regress the saved model.
    if CRT_EPOCHS > 0:
        log_line("=== Phase 3: classifier re-training (cRT, class-balanced) ===")
        model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
        crt_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE,
            sampler=balanced_sampler(train_samples, len(classes)),
            num_workers=NUM_WORKERS, pin_memory=_pin, persistent_workers=NUM_WORKERS > 0,
        )
        crt_criterion = nn.CrossEntropyLoss()  # sampler balances classes, so no weights
        freeze_backbone(model, freeze=True)
        opt = torch.optim.AdamW(head_parameters(model), lr=CRT_LR)
        best = train_epochs(model, crt_loader, val_loader, crt_criterion, opt, CRT_EPOCHS,
                            "crt", history, scaler, best=best)

    log_line(f"Training done. Best val balanced accuracy: {best:.3f}")

## 8. Final evaluation & visualization

In [ ]:
# Load best checkpoint and evaluate
model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=True))
model.to(device)
preds, targets, bal_acc = evaluate(model, val_loader)
print(f"Final validation balanced accuracy: {bal_acc:.3f}")

In [ ]:
# Confusion matrix (row-normalized)
cm = confusion_matrix(targets, preds, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(classes)), classes, fontsize=9)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("Predicted", fontsize=10)
ax.set_ylabel("True", fontsize=10)
ax.set_title(f"Confusion matrix (val, row-normalized) — bal-acc {bal_acc:.3f}", fontsize=11)
fig.colorbar(im)
fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved confusion matrix to {OUT_DIR / 'confusion_matrix.png'}")

In [ ]:
# Training curves. x-axis is a running epoch index across all phases (head / ft /
# crt), with a dashed divider each time the phase changes.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
steps = list(range(len(history)))
boundaries = [i for i in range(1, len(history)) if history[i]["phase"] != history[i - 1]["phase"]]

ax1.plot(steps, [h["loss"] for h in history], marker="o", linewidth=2)
for b in boundaries:
    ax1.axvline(b - 0.5, color="red", linestyle="--", alpha=0.5)
ax1.set_xlabel("Epoch (all phases)"); ax1.set_ylabel("Training Loss"); ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(steps, [h["val_bal_acc"] for h in history], marker="o", linewidth=2, color="green")
for b in boundaries:
    ax2.axvline(b - 0.5, color="red", linestyle="--", alpha=0.5)
ax2.set_xlabel("Epoch (all phases)"); ax2.set_ylabel("Validation Balanced Accuracy")
ax2.set_title("Validation Balanced Accuracy"); ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved training curves to {OUT_DIR / 'training_curves.png'}")

## 9. Save results

In [ ]:
with open(OUT_DIR / "training_log.json", "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "img_size": IMG_SIZE,
        "classes": list(classes),
        "best_val_bal_acc": float(best),
        "history": history,
    }, f, indent=2)

with open(OUT_DIR / "classes.json", "w") as f:
    json.dump(list(classes), f)

print(f"All results saved to {OUT_DIR}:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")
print("\nClick 'Save Version' (top-right) to persist /kaggle/working, then download")
print("the contents of results/ from the version's Output tab into the repo's")
print("skin/results/ directory — skin/03_app.py reads model_config.json and adapts.")
# --- close out the wandb run -------------------------------------------------
if USE_WANDB and wandb is not None and wandb.run is not None:
    wandb.run.summary["best_val_bal_acc"] = float(best)
    for _f in ("confusion_matrix.png", "training_curves.png"):
        if (OUT_DIR / _f).exists():
            wandb.log({_f[:-4]: wandb.Image(str(OUT_DIR / _f))})
    wandb.finish()
    print("wandb: run finished and synced.")


## 10. Free GPU memory before trying another architecture

Run this **only** when you want to train a different `MODEL_NAME` in the same session.
It releases the model/optimizer/loaders and clears the CUDA cache so a second backbone
doesn't stack on the first and OOM. Afterwards, change `MODEL_NAME` in the config cell and
re-run from **section 4 (Data loaders)** onward.

In [ ]:
import gc

for _name in ["model", "opt", "criterion", "crt_criterion", "train_loader", "crt_loader",
              "val_loader", "train_ds", "val_ds", "preds", "targets", "scaler"]:
    if _name in globals():
        del globals()[_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Freed model/loaders and cleared the CUDA cache.")
print("Now change MODEL_NAME in the config cell and re-run from section 4 (Data loaders).")